# Mirage — tiny on-device vision and structured extraction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Mirage-tiny-vision-structured-extraction.ipynb)

[`Cactus-Compute/mirage`](https://huggingface.co/Cactus-Compute/mirage) is Cactus Compute's Apache-2.0 vision model for phones, wearables, and other consumer devices. The [blog](https://cactuscompute.com/blog/mirage) describes **51.3 million parameters** packed into about **22 MB** (`mirage.cact` is **21.6 MB** of 2-bit Cactus Quants). A companion `mirage.safetensors` on the Hub is **216 MB**; the Python API loads the `.cact` file, not that checkpoint.

**Why this model.** One forward pass returns a **structured record** — scene description, open-vocab objects with boxes, OCR with a score, segmentation masks, spatial relations, and schema-based decisions — instead of a free-text caption. The card's usage is:

```sh
pip install -U cactus-compute
```

```python
import cactus
print(cactus.Mirage().describe("photo.png"))
```

See [Use cases](https://github.com/donvito/notebooks/blob/main/docs/Mirage-tiny-vision-structured-extraction.md) for when this 22 MB model is a good fit and when it is not.

**Licence:** Apache-2.0. Public, not gated. `HF_TOKEN` is optional.

**Runtime:** *Runtime > Change runtime type > T4 GPU* is the runtime this notebook is tested on, but **Mirage does not use the GPU**. Inference is the Cactus **CPU** engine (Linux x86-64 wheel from [`Cactus-Compute/cactus`](https://huggingface.co/Cactus-Compute/cactus)). A free CPU runtime is enough. The next cells still print `torch.cuda.is_available()`, the device name, and free VRAM. First use downloads the engine plus `mirage.cact` (~22 MB).

The blog quotes **74 ms** on a Samsung Galaxy S26 NPU and **107 ms** on that phone's CPU. Those are vendor phone numbers, not Colab measurements. This notebook prints wall time per image on *this* runtime.

**Not in this notebook:** native `cactus_load` / `cactus_describe` C bindings, browser WASM, or pairing Mirage JSON with a separate text LLM (the blog's small VQAv2 exploration).

**Links:** [model card](https://huggingface.co/Cactus-Compute/mirage) · [blog](https://cactuscompute.com/blog/mirage) · [Python API](https://cactuscompute.com/blog/needle-python-docs) · [runtime](https://huggingface.co/Cactus-Compute/cactus) · [source](https://github.com/cactus-compute/cactus) · [announcement](https://x.com/cactuscompute/status/2118761482343313844) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/Mirage-tiny-vision-structured-extraction.md)


## 1. Install `cactus-compute` (keep Colab's torch)

The card is `pip install -U cactus-compute`. This cell pins **4.2.2**, the PyPI release the notebook was checked against. The wheel pulls `huggingface_hub` and Pillow. It does **not** install or replace PyTorch.

Use **`%pip`**, not `python -I -m pip`. Isolated mode drops Colab site-packages. Do **not** `pip install torch`.


In [ ]:
%pip install -q -U "cactus-compute==4.2.2"

import sys

print("python", sys.version.split()[0])
print("install done — cactus-compute does not replace Colab's torch")


## 2. Runtime check

Print `torch.cuda.is_available()`, the GPU name, and free VRAM even though Mirage runs on the **CPU** engine. A missing GPU is fine.


In [ ]:
import os
import shutil

os.environ["CACTUS_TELEMETRY"] = "0"
os.environ["DO_NOT_TRACK"] = "1"

import cactus
import torch

print("cactus-compute", cactus.__version__)
print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    major, minor = torch.cuda.get_device_capability()
    print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
    print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
else:
    print("GPU name: none")
    print("VRAM free: n/a")

print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")
print("Mirage inference uses the Cactus CPU engine (not CUDA). A T4 is unused for this model.")


## 3. Load Mirage

`cactus.Mirage()` fetches the engine and `mirage.cact` on first use, then caches them under `~/.cache/cactus/`. Reuse the instance. `describe` accepts a path, encoded bytes, a Pillow image, or an RGB `uint8` array. Boxes are on a **0–1000** scale relative to the image ([blog](https://cactuscompute.com/blog/mirage)).


In [ ]:
import json
import time
from pathlib import Path

import cactus

t0 = time.perf_counter()
vision = cactus.Mirage()
print(f"load wall {time.perf_counter() - t0:.2f}s")
print("weights", vision.weights)


def mirage_describe(image, **kwargs):
    """Call describe(); replace non-JSON scores (NaN / Infinity) if the engine emits them."""
    try:
        return vision.describe(image, **kwargs)
    except json.JSONDecodeError:
        raw = vision._buffer.value.decode("utf-8", "replace")
        cleaned = (
            raw.replace("-nan", "null")
            .replace("nan", "null")
            .replace("-Infinity", "null")
            .replace("Infinity", "null")
        )
        print("note: engine emitted a non-JSON number (e.g. NaN score); replaced with null")
        return json.loads(cleaned)


def mirage_extract(image, form, **kwargs):
    try:
        return vision.extract(image, form, **kwargs)
    except json.JSONDecodeError:
        raw = vision._buffer.value.decode("utf-8", "replace")
        cleaned = (
            raw.replace("-nan", "null")
            .replace("nan", "null")
            .replace("-Infinity", "null")
            .replace("Infinity", "null")
        )
        print("note: engine emitted a non-JSON number (e.g. NaN score); replaced with null")
        return json.loads(cleaned)


## 4. Public sample images

| Sample | Why | Source |
| --- | --- | --- |
| COCO `000000000139` | indoor scene for `describe()` + boxes | [COCO val2017](http://images.cocodataset.org/val2017/000000000139.jpg) |
| COCO `000000231049` | stop sign used in the [Mirage blog](https://cactuscompute.com/blog/mirage) for OCR + masks | COCO val2017 / [blog JPEG](https://cactuscompute.com/mirage/coco-231049.jpg) |
| SROIE receipt | printed receipt for `extract()` | Hugging Face OCR fixtures (same file the LightOnOCR notebook uses) |

Retry on 429/5xx. Boxes and polygon masks are 0–1000; overlays convert them to pixels.


In [ ]:
import io
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import requests

SAMPLE_DIR = Path("mirage_samples")
SAMPLE_DIR.mkdir(exist_ok=True)
UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}

SCENE_URLS = [
    "http://images.cocodataset.org/val2017/000000000139.jpg",
]
SIGN_URLS = [
    "https://cactuscompute.com/mirage/coco-231049.jpg",
    "http://images.cocodataset.org/val2017/000000231049.jpg",
]
RECEIPT_URLS = [
    "https://huggingface.co/datasets/hf-internal-testing/fixtures_ocr/resolve/main/SROIE-receipt.jpeg",
]
PALETTE = [
    (31, 119, 180),
    (255, 127, 14),
    (44, 160, 44),
    (214, 39, 40),
    (148, 103, 189),
    (140, 86, 75),
    (227, 119, 194),
    (127, 127, 127),
]


def fetch_bytes(url, attempts=4):
    for attempt in range(attempts):
        response = requests.get(url, headers=UA, timeout=120)
        if response.status_code not in (429, 500, 502, 503, 504) or attempt == attempts - 1:
            response.raise_for_status()
            return response.content
        retry_after = response.headers.get("Retry-After", "")
        time.sleep(int(retry_after) if retry_after.isdigit() else 2 ** (attempt + 1))


def fetch_to(path, *urls):
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        return path
    last_error = None
    for url in urls:
        try:
            path.write_bytes(fetch_bytes(url))
            print("saved", path.name, f"({path.stat().st_size} bytes)")
            return path
        except requests.RequestException as error:
            last_error = error
            print(f"Could not fetch {url}: {error}")
    raise RuntimeError(f"All URLs failed for {path.name}: {last_error}")


def load_image(path):
    return Image.open(path).convert("RGB")


def show_image(image, title=None, longest=720):
    preview = image.copy()
    preview.thumbnail((longest, longest))
    if title:
        print(title, image.size, "-> preview", preview.size)
    display(preview)


def bbox_xyxy(bbox, width, height):
    x1, y1, x2, y2 = bbox
    return (
        x1 / 1000.0 * width,
        y1 / 1000.0 * height,
        x2 / 1000.0 * width,
        y2 / 1000.0 * height,
    )


def ring_to_pixels(ring, width, height):
    pts = []
    for i in range(0, len(ring) - 1, 2):
        pts.append((ring[i] / 1000.0 * width, ring[i + 1] / 1000.0 * height))
    return pts


def mask_rings(mask):
    if mask is None:
        return []
    if isinstance(mask, dict):
        return []  # RLE bitmap; overlays below use polygons
    if not mask:
        return []
    if isinstance(mask[0], (int, float)):
        return [mask]
    return mask


def draw_boxes(image, items, label_key="name", score_key="score"):
    overlay = image.copy()
    draw = ImageDraw.Draw(overlay)
    try:
        font = ImageFont.load_default()
    except Exception:
        font = None
    w, h = image.size
    for i, item in enumerate(items):
        bbox = item.get("bbox")
        if not bbox or len(bbox) < 4:
            continue
        color = PALETTE[i % len(PALETTE)]
        x1, y1, x2, y2 = bbox_xyxy(bbox, w, h)
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        label = str(item.get(label_key) or item.get("string") or i)
        score = item.get(score_key)
        if isinstance(score, (int, float)):
            label = f"{label} {score:.0%}"
        draw.text((x1 + 2, max(0, y1 - 12)), label, fill=color, font=font)
    return overlay


def draw_masks(image, objects, alpha=96):
    overlay = image.convert("RGBA")
    layer = Image.new("RGBA", overlay.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(layer)
    w, h = overlay.size
    drawn = 0
    for i, obj in enumerate(objects):
        color = PALETTE[i % len(PALETTE)] + (alpha,)
        for ring in mask_rings(obj.get("mask")):
            pts = ring_to_pixels(ring, w, h)
            if len(pts) >= 3:
                draw.polygon(pts, fill=color, outline=color[:3] + (220,))
                drawn += 1
    return Image.alpha_composite(overlay, layer).convert("RGB"), drawn


def show_overlay(image, title=None):
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image)
    ax.set_axis_off()
    if title:
        ax.set_title(title)
    plt.show()
    plt.close(fig)


def compact(result):
    """JSON-safe copy without bulky mask payloads."""
    out = json.loads(json.dumps(result, default=str))
    for obj in out.get("objects") or []:
        if "mask" in obj:
            rings = mask_rings(obj["mask"])
            obj["mask"] = f"<{len(rings)} ring(s)>"
    return out


scene_path = fetch_to(SAMPLE_DIR / "coco-000139.jpg", *SCENE_URLS)
sign_path = fetch_to(SAMPLE_DIR / "coco-231049.jpg", *SIGN_URLS)
receipt_path = fetch_to(SAMPLE_DIR / "sroie-receipt.jpeg", *RECEIPT_URLS)

scene = load_image(scene_path)
sign = load_image(sign_path)
receipt = load_image(receipt_path)
show_image(scene, "COCO 000000000139")
show_image(sign, "COCO 000000231049 (stop sign)")
show_image(receipt, "SROIE receipt")


## 5. Demo 1 — `describe()` with object boxes

Card usage: `cactus.Mirage().describe("photo.png")`. This cell keeps objects and boxes (`fields=["bbox", "score"]`) so the overlay is readable. Coordinates are 0–1000.


In [ ]:
t0 = time.perf_counter()
scene_result = mirage_describe(scene_path, sections=["objects"], fields=["bbox", "score"])
scene_s = time.perf_counter() - t0

objects = scene_result.get("objects") or []
print(f"wall {scene_s:.2f}s | objects {len(objects)}")
print(json.dumps(compact(scene_result), indent=2)[:2500])
print("names:", [obj.get("name") for obj in objects])
boxed = draw_boxes(scene, objects)
show_overlay(boxed, f"describe() boxes — {scene_s:.2f}s")


## 6. Demo 2 — OCR + segmentation masks

The [blog](https://cactuscompute.com/blog/mirage) walks through this COCO stop-sign photo: objects, masks, and the OCR string **ARRÉT** (it notes the incorrect accent on French *ARRÊT*). `mask_format="polygons"` is the documented outline format. Text lines include `string`, `bbox`, and `score`.


In [ ]:
t0 = time.perf_counter()
sign_result = mirage_describe(
    sign_path,
    sections=["objects", "text"],
    fields=["bbox", "score", "mask"],
    mask_format="polygons",
)
sign_s = time.perf_counter() - t0

sign_objects = sign_result.get("objects") or []
sign_text = sign_result.get("text") or []
print(f"wall {sign_s:.2f}s | objects {len(sign_objects)} | text lines {len(sign_text)}")
print("OCR:")
for line in sign_text:
    score = line.get("score")
    score_s = f"  {score:.3f}" if isinstance(score, (int, float)) else ""
    print(f"  {line.get('string')!r}{score_s}  bbox={line.get('bbox')}")
print("objects:", [(obj.get("name"), obj.get("score")) for obj in sign_objects])

masked, n_rings = draw_masks(sign, sign_objects)
print(f"polygon rings drawn: {n_rings}")
ocr_boxes = draw_boxes(sign, sign_text, label_key="string")
show_overlay(ocr_boxes, f"OCR boxes — {sign_s:.2f}s")
show_overlay(masked, f"object masks (polygons) — {sign_s:.2f}s")


## 7. Demo 3 — schema extract / visual decision

`extract(image, form)` fills ordered fields. The [Python API](https://cactuscompute.com/blog/needle-python-docs) documents `enum`, bounded `integer`, `string`, and box references. The blog's sign example is `{"sign": {"enum": ["stop", "yield", "speed limit"]}}`. Here the SROIE receipt gets a `document_type` choice. `form[field]` is `{value, probabilities}`; `value` may be `None` when the engine abstains (`"-"`).


In [ ]:
RECEIPT_FORM = {
    "document_type": {"enum": ["receipt", "letter", "photo", "sign"]},
}

t0 = time.perf_counter()
receipt_result = mirage_extract(receipt_path, RECEIPT_FORM)
receipt_s = time.perf_counter() - t0

print(f"wall {receipt_s:.2f}s")
print(json.dumps(receipt_result.get("form"), indent=2))
form = receipt_result.get("form") or {}
choice = (form.get("document_type") or {}).get("value")
print("selected:", choice)

SIGN_FORM = {"sign": {"enum": ["stop", "yield", "speed limit"]}}
t0 = time.perf_counter()
sign_form = mirage_extract(sign_path, SIGN_FORM)
sign_form_s = time.perf_counter() - t0
print(f"\nblog sign extract wall {sign_form_s:.2f}s")
print(json.dumps(sign_form.get("form"), indent=2))

print("\nwall time summary")
print(f"  COCO indoor describe : {scene_s:.2f}s")
print(f"  stop-sign OCR+masks  : {sign_s:.2f}s")
print(f"  receipt extract      : {receipt_s:.2f}s")
print(f"  stop-sign extract    : {sign_form_s:.2f}s")


## 8. Caveats and cleanup

- **CPU engine.** A T4 (or any CUDA GPU) is not used. The GPU check is informational. The blog's 74 ms NPU / 107 ms CPU figures are for a Galaxy S26, not Colab.
- **~22 MB runtime weights.** `mirage.cact` is 21.6 MB. `mirage.safetensors` (~216 MB) is on the card and is not loaded here.
- **NaN object scores.** The engine can emit `-nan` in JSON for a detection score. The helper replaces those with `null` so `json.loads` succeeds. Text-line scores on the stop-sign demo were ordinary floats.
- **OCR accents.** The blog's own overlay on this stop-sign photo shows **ARRÉT** and calls out the incorrect accent. That is a documented miss, not a Colab glitch. Do not treat this as a production OCR stack.
- **Language list.** The card lists **en, de, fr, es, it, nl, pl**. Pass `language="de"` (and so on) for a description in that language.
- **Extract abstains.** A field can come back with `"value": null` and mass on `"-"`. That is the engine declining a choice.
- **Not a chat VLM.** There is no free-text Q&A loop. For that, use [MiniCPM-V](MiniCPM-V-4.7-1B-vision.ipynb) or [LFM3-VL](LFM3-VL-2B-vision.ipynb). For dedicated OCR, use [LightOnOCR](LightOnOCR-3-0.8B-document-ocr.ipynb) or [Indra OCR](Indra-OCR-1M-document-ocr.ipynb).
- **Telemetry.** Anonymous package counts can be switched off with `CACTUS_TELEMETRY=0` or `DO_NOT_TRACK=1` (already set above).
- **Apache-2.0.** Unrestricted commercial and research use, subject to the licence text.

Drop the instance before opening another notebook:


In [ ]:
import gc

print("weights still at", getattr(vision, "weights", None))
if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    print(f"VRAM free / total (unused by Mirage): {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")

try:
    del vision
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Mirage released. Re-run the load cell if you want another image.")
